<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Edges vs Command — 何时该用谁

**一句话定位**:`Command` 是 LangGraph 的「**升级版节点返回值**」—— 把「**更新 state**」和「**决定下一节点**」**二合一**,省掉条件边的中间转译。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**官方文档原话**:

> `Command` objects in LangGraph can both **update the state** and **select the next node to visit**. This is a useful alternative to edges.

**说人话**:

- ❌ **老路**:节点 `return {...state 更新...}` → **再**写一个 `conditional_edges` 函数从 state 读出决策、决定去哪
- ✅ **新路**:节点直接 `return Command(update={...}, goto="next_node")` —— 一次返回,**state 和路径同时定**

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔑 一图看明白核心区别

</div>

| 维度 | 🔵 Edges(传统) | 🟡 Command(新) |
|------|-----------------|------------------|
| 节点返回值 | `dict`(只更 state) | `Command(update=..., goto=...)` |
| 路径决定权 | **图构建时** 写死 | **节点运行时** 现算 |
| 看路径要看哪 | `add_edge` / `add_conditional_edges` | 节点函数内部 |
| 跨子图跳转 | ❌ 不行 | ✅ `Command(graph=Command.PARENT, ...)` |
| 可视化清晰度 | 高(`get_graph()` 一眼看清) | 中(动态路由,需类型注解才画出) |

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># 🔵 Edges 写法 ────────────────────────────
def call_llm(state):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}                # 只更 state

def route(state) -> Literal["tool", END]:           # ← 额外一个函数
    return "tool" if state["messages"][-1].tool_calls else END

graph.add_conditional_edges("call_llm", route)

# 🟡 Command 写法 ──────────────────────────
def call_llm(state) -> <span style="background:#3d3414; color:#fde047; padding:0 2px;">Command[Literal["tool", "__end__"]]</span>:
    response = llm.invoke(state["messages"])
    next_node = "tool" if response.tool_calls else END
    return <span style="background:#3d3414; color:#fde047; padding:0 2px;">Command(update={"messages": [response]}, goto=next_node)</span>

# 不需要 add_conditional_edges
</code></pre>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📊 顺便复习:Edges 的 3 种基础形态

</div>

| 形态 | 写法 | 适用场景 |
|------|------|----------|
| **起点边** | `add_edge(START, "a")` | 入口固定 |
| **直连边** | `add_edge("a", "b")` | 永远从 a 到 b,无分支 |
| **条件边** | `add_conditional_edges("a", fn)` | 多个候选,**运行时挑** |

**conditional_edges 函数签名**:`(state) -> str | list[str]`,返回**下一个 node 的名字**(返回 list 则**并行 fan-out**)。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察:Command 到底解决了什么疼点?**

用 `conditional_edges` 做动态路由,你的逻辑**散在 3 处**:

1. 节点函数算出 decision
2. 把 decision **写进 state**
3. `conditional_edges` 函数从 state **再读出来**,return next_node

`Command` 把这 3 步**压成 1 步** —— decision 算出来直接 `goto`,state 里**都不一定要存它**。

👉 一条判断准则:**当「路由决定 = 节点刚算出的结果」时,Command 是直接胜利。**

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎯 何时用 Edges?何时用 Command?

</div>

| 你的场景 | 🔵 Edges | 🟡 Command |
|----------|:--------:|:----------:|
| 入口边(`START → 第一个 node`) | ✅ | |
| 直连(永远 a → b,无分支) | ✅ | |
| 路由依据已在 state 里(**别的节点**写的) | ✅ | |
| 路由决定 **就是当前节点刚算出的** | | ✅ |
| 同时要 **update state + goto** | | ✅ |
| 多个并行分支(fan-out 给多个 node) | ✅ | ✅ (用 `goto=["a","b"]`) |
| **子图节点要跳到父图节点** | ❌ 做不到 | ✅ `graph=Command.PARENT` |
| 重视图可视化(`get_graph()` 拓扑) | ✅ | |
| 重视代码精简、避免「写→读 state」往返 | | ✅ |

**🎤 口诀**:

> **路由信息只属于「此刻这个节点」→ Command。**
>
> **路由信息属于「整张图的拓扑」→ Edges。**

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔬 真实案例 — Email assistant 的 triage router

</div>

**任务**:LLM 分类邮件「Respond / Notify / Ignore」,跳到对应分支。

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**❌ 传统 edges 写法(冗余,3 处碰同一信息)**

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">class State(MessagesState):
    classification_decision: str   # ← ① 多搞一个字段

def triage(state):
    result = router_llm.invoke(state["email_input"])
    return {"classification_decision": result.classification}   # ② 写 state

def route_after_triage(state):
    if state["classification_decision"] == "respond":           # ③ 又读 state
        return "response_agent"
    elif state["classification_decision"] == "notify":
        return "triage_interrupt_handler"
    return END

graph.add_node("triage", triage)
graph.add_conditional_edges("triage", route_after_triage)
</code></pre>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✅ Command 写法(精炼,1 处搞定)**

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">def triage(state) -> <span style="background:#3d3414; color:#fde047; padding:0 2px;">Command[Literal["response_agent", "triage_interrupt_handler", "__end__"]]</span>:
    result = router_llm.invoke(state["email_input"])
    next_node = {
        "respond": "response_agent",
        "notify":  "triage_interrupt_handler",
        "ignore":  END,
    }[result.classification]
    return <span style="background:#3d3414; color:#fde047; padding:0 2px;">Command(
        update={"classification_decision": result.classification},
        goto=next_node,
    )</span>

graph.add_node("triage", triage)   # ← 不需要 add_conditional_edges
</code></pre>

**省了一个函数、一处「写 → 读 state」往返、一个零散 edge 声明** —— 全部逻辑**待在一个节点里**。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚀 Command 独有的两个能力(Edges 做不到)

</div>

**1️⃣ 子图节点直接跳到父图节点 —— `Command.PARENT`**

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">def subagent_finish(state):
    ...
    return Command(
        <span style="background:#3d3414; color:#fde047; padding:0 2px;">graph=Command.PARENT</span>,        # ← 跳出当前子图
        goto="parent_node_name",      # 跳到父图的某节点
        update={"shared_state_key": "..."},
    )
</code></pre>

用 edges 你只能在父图里加 `add_edge(subgraph_node, parent_node)`,**做不到从子图节点动态选父图节点**。

**2️⃣ 类型注解仍能让 `get_graph()` 画出箭头**

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">def node(state) -> Command[<span style="background:#3d3414; color:#fde047; padding:0 2px;">Literal["a", "b", END]</span>]:
    ...
</code></pre>

**只要注解写了**,LangGraph 会把这些候选边画出来 —— Command **不是完全黑盒**。漏写注解则图上不显示动态分支(运行不影响,只是不好看)。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**⚠️ 易踩坑**

- `goto="xxx"` 里的名字必须是**已经 `add_node` 过的** —— 拼错只在**运行时**报错,不在编译时
- `update={...}` 必须**字段已在 state schema 里** —— 不在的字段会被忽略(不报错!)
- 不要在一个节点里同时既 return `Command` 又依赖 `add_edge` —— 行为不可预测,选一种风格

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Edges = 图拓扑(静态、声明在外);Command = 节点决策(动态、嵌在节点内)。**

🎯 **90% 简单图用 edges 就够**;**只有当「路由决定就是节点算出的结果」或「需要跨子图跳转」时,才用 Command**。

⚠️ 别为了用 Command 而用 Command —— 该用 edges 就用 edges,可视化更清晰、维护更直白。

</div>